# VP-Skin: reproducible Kaggle notebook (AID 836, PS2)
Baseline: GauHuman @ `732ec0d` · Ours: volume-preserving rigid skinning + unseen-pose regularisation.

**Accelerator:** GPU T4 (16 GB). One GPU is enough.

**Inputs:** attach these as *private* Kaggle datasets. Licences forbid redistribution, so each user creates them from their own downloads (see README):
- `smpl-models/SMPL_NEUTRAL.pkl`, already cleaned with `tools/clean_smpl.py`
- `zju-mocap-refine/` (subjects my_377 … my_394)
- optional `vpskin-wheels/` (prebuilt CUDA wheels from an earlier run, for a fast install)

Every train/eval run appends peak VRAM, wall-clock and GPU-hours to `results/runs.csv`.

In [ ]:
# ---- configuration ----
MODE      = "smoke"     # "smoke" (tests + 1 short run) | "train_one" | "full"
REPO_URL  = "https://github.com/Harsha081459/vpskin-gs.git"   # fork of skhu101/GauHuman
BRANCH    = "vpskin"
SUBJECTS  = ["my_377", "my_386", "my_387", "my_392", "my_393", "my_394"]
SUBJECT   = "my_377"
ITERS     = 1200        # GauHuman default for ZJU-MoCap-refine
INPUT     = "/kaggle/input"
WORK      = "/kaggle/working"

In [ ]:
# ---- environment report (goes into the report's reproducibility section) ----
import subprocess, sys, torch
print(sys.version)
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
print(subprocess.run("nvcc --version | tail -2", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%cd {WORK}
!git clone -q -b {BRANCH} {REPO_URL} vpskin-gs || (cd vpskin-gs && git pull -q)
%cd {WORK}/vpskin-gs
!git log --oneline -1

In [ ]:
# ---- install: reuse cached wheels if attached, otherwise build once (~10 min) and save them ----
import glob, os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"          # T4
cached = glob.glob(f"{INPUT}/vpskin-wheels/*.whl")
if cached:
    !pip install -q {" ".join(cached)}
else:
    !pip wheel -q --no-build-isolation --no-deps submodules/diff-gaussian-rasterization submodules/simple-knn -w {WORK}/wheels
    !pip install -q {WORK}/wheels/*.whl
!pip install -q plyfile lpips==0.1.4
# knn_cuda is optional: scene/gaussian_model.py falls back to vpskin/knn_fallback.py
import diff_gaussian_rasterization, simple_knn; print("CUDA extensions OK")

In [ ]:
# ---- data links ----
os.makedirs("assets", exist_ok=True); os.makedirs("data", exist_ok=True)
!ln -sf {INPUT}/smpl-models/SMPL_NEUTRAL.pkl assets/SMPL_NEUTRAL.pkl
!ln -sfn {INPUT}/zju-mocap-refine data/zju_mocap_refine
!ls assets data/zju_mocap_refine | head

In [ ]:
# ---- unit tests (skinning maths, regularisers, logger) ----
!python -m pytest -q tests

## Baseline B0: GauHuman on ZJU-MoCap-refine
Split: train on view 4, frames 0–495 step 5. Test on 22 views × 17 frames. Paper target (average of 6 subjects): PSNR 31.34 / SSIM 0.965 / LPIPS\* 30.51.

In [ ]:
def train_eval(subject, exp, extra=""):
    ds = f"data/zju_mocap_refine/{subject}"
    it = 300 if MODE == "smoke" else ITERS
    !python train.py -s {ds} --eval --exp_name {exp} --motion_offset_flag --smpl_type smpl --actor_gender neutral --iterations {it} {extra}
    !python render.py -m output/{exp} --motion_offset_flag --smpl_type smpl --actor_gender neutral --iteration {it} --skip_train

subjects = [SUBJECT] if MODE in ("smoke", "train_one") else SUBJECTS
for s in subjects:
    train_eval(s, f"zju_mocap_refine/B0_{s}")

In [ ]:
# ---- results table: image metrics + cost ----
import json, glob, pandas as pd
rows = []
for f in glob.glob("output/zju_mocap_refine/*/metrics_test_*.json"):
    d = json.load(open(f)); d["exp"] = f.split("/")[-2]; rows.append(d)
df = pd.DataFrame(rows)
display(df[["exp", "psnr", "ssim", "lpips_x1000", "fps_sync", "n_gaussians"]].round(3) if len(df) else df)
runs = pd.read_csv("results/runs.csv")
display(runs[["run", "gpu", "wall_clock_s", "gpu_hours", "peak_alloc_mb", "peak_reserved_mb", "status"]])
print("total GPU-hours:", runs.gpu_hours.sum().round(3))

In [ ]:
# ---- keep artefacts: save wheels + results to /kaggle/working (export with 'Save Version') ----
!mkdir -p {WORK}/export && cp -r results {WORK}/export/ && cp -r {WORK}/wheels {WORK}/export/ 2>/dev/null; ls {WORK}/export